# 03. Supervised Model Training & Model Comparison — ResumeAI
Vectorizes text with TF-IDF and trains Logistic Regression, Naive Bayes, Linear SVM, and Random Forest.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import joblib

## 1. Load Processed Data & Split

In [ ]:
df = pd.read_csv('../data/processed/cleaned_resumes.csv')
X = df['Cleaned_Resume'].astype(str).values
y = df['Category'].astype(str).values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f'Train: {len(X_train)}, Validation: {len(X_val)}')

## 2. TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=3500, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')

## 3. Train and Compare Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Multinomial NB': MultinomialNB(alpha=0.1),
    'Linear SVM': CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=42), cv=3),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_val_vec)
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro', zero_division=0)
    results.append({'Model': name, 'Accuracy': acc, 'Macro F1': f1})

res_df = pd.DataFrame(results).sort_values(by='Macro F1', ascending=False)
res_df